# Estatística — formulário comentado e calculadora Python

**Material de consulta para provas** · Conteúdo organizado na ordem da disciplina.

**Como usar:** leia a seção **Quando usar**, confira as hipóteses da fórmula e **altere apenas os dados de entrada** nas células Python. Execute com `Shift + Enter`. Todas as fórmulas são escritas em LaTeX, renderizado pelo Jupyter.

> **Atenção:** este material cobre o núcleo usual dos seis assuntos, mas não substitui a ementa e os métodos exigidos pelo professor. Alguns testes e distribuições podem ter parametrizações diferentes em outros livros. Os exemplos são ilustrativos, não respostas a questões específicas.

## Índice
1. Análise de Dados Quantitativos
2. Probabilidades
3. Variáveis Aleatórias Discretas Unidimensionais
4. Variáveis Aleatórias Contínuas Unidimensionais
5. Testes de Hipótese
6. Modelo Básico de Regressão Linear
7. **Dicionário de notações**

**Bibliotecas necessárias:** `numpy scipy pandas matplotlib statsmodels sympy`. Instale no ambiente virtual do projeto com `python -m pip install numpy scipy pandas matplotlib statsmodels sympy`.

In [ ]:
import numpy as np
import pandas as pd
from fractions import Fraction
from scipy import stats, integrate, special
import statsmodels.api as sm
import matplotlib.pyplot as plt

print("Bibliotecas carregadas. Pronto para calcular!")

---
# 1. Análise de Dados Quantitativos

**Identificação:** quando a questão fornece observações numéricas e pede resumo, posição, dispersão, comparação ou associação.

### Média, mediana, moda e amplitude

**Quando usar:** resumir a posição central e a extensão de um conjunto de observações.

**Fórmula(s):**

$$\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i \qquad A=\max(x_i)-\min(x_i)$$

**Cuidado:** a moda pode ser múltipla; a função abaixo lista todas as modas. Dados numéricos em lista.

In [ ]:
dados = np.array([2, 4, 4, 6, 9], dtype=float)
print("Média:", np.mean(dados))
print("Mediana:", np.median(dados))
print("Moda(s):", stats.mode(dados, keepdims=False).mode, "(moda principal via SciPy)")
print("Amplitude:", np.ptp(dados))

### Média ponderada

**Quando usar:** os valores têm pesos diferentes (frequências, notas ou probabilidades).

**Fórmula(s):**

$$\bar{x}_w=\frac{\sum_{i=1}^{n}w_ix_i}{\sum_{i=1}^{n}w_i}$$

Se os pesos são probabilidades, devem somar 1 e o resultado é a esperança discreta.

In [ ]:
valores = np.array([5, 7, 10], dtype=float)
pesos = np.array([1, 2, 1], dtype=float)
print("Média ponderada:", np.average(valores, weights=pesos))

### Variância e desvio-padrão

**Quando usar:** medir dispersão; distinguir amostra de população.

**Fórmula(s):**

$$\sigma^2=\frac{\sum(x_i-\mu)^2}{N},\qquad s^2=\frac{\sum(x_i-\bar{x})^2}{n-1}$$

$$\sigma=\sqrt{\sigma^2},\qquad s=\sqrt{s^2}$$

**`ddof=0`**: população; **`ddof=1`**: variância/desvio-padrão amostrais (correção de Bessel).

In [ ]:
dados = np.array([4, 6, 8, 10], dtype=float)
print("Variância populacional:", np.var(dados, ddof=0))
print("Variância amostral:", np.var(dados, ddof=1))
print("Desvio-padrão amostral:", np.std(dados, ddof=1))

### Quartis, percentis e intervalo interquartil

**Quando usar:** identificar posições e dispersão robusta.

**Fórmula(s):**

$$AIQ=Q_3-Q_1$$

**Atenção:** livros e softwares usam convenções diferentes de interpolação para quartis; confirme o método da disciplina.

In [ ]:
dados = np.array([2, 3, 5, 7, 8, 10, 12, 15], dtype=float)
q1, q2, q3 = np.percentile(dados, [25, 50, 75], method="linear")
print("Q1, mediana, Q3:", q1, q2, q3)
print("AIQ:", q3 - q1)
print("Percentil 90:", np.percentile(dados, 90))

### Gerar um bloxplot a partir de uma lista de elementos

A linha laranja no meio é a mediana, o lado de cima do retângulo é o terceiro quartil, o lado de baixo é o primeiro quartil e os traços após as linhas verticais são os últimos elementos não considerados outliers.

In [ ]:
dados = [60, 80, 80, 85, 85, 85, 85, 90, 90, 90, 90, 90, 100, 100, 100, 100, 100, 100]
plt.boxplot(dados)
plt.title('Boxplot de Dados Fictícios')
plt.ylabel('Valores')
plt.show()

### Coeficiente de variação e escore padronizado

**Quando usar:** comparar dispersões relativas (médias positivas e não próximas de zero) ou padronizar valores.

**Fórmula(s):**

$$CV=\frac{s}{\bar{x}}\times100\%,\qquad z=\frac{x-\mu}{\sigma}$$

CV pode ser inadequado se a média é zero, muito próxima de zero ou a escala não admite razão significativa.

In [ ]:
dados = np.array([10, 12, 14, 16], dtype=float)
cv = np.std(dados, ddof=1) / np.mean(dados) * 100
x, mu, sigma = 120, 100, 15
print("CV (%):", cv)
print("Escore z:", (x - mu) / sigma)

### Frequências e tabelas

**Quando usar:** contar observações por valor e obter frequências relativas/acumuladas.

**Fórmula(s):**

$$f_i=\#\{x_j=a_i\},\qquad fr_i=\frac{f_i}{n}$$

Para variáveis contínuas agrupadas em classes, escolha intervalos explicitamente.

In [ ]:
dados = pd.Series([1, 1, 2, 2, 2, 3, 4])
freq = dados.value_counts().sort_index()
tabela = pd.DataFrame({"frequência": freq, "relativa": freq / len(dados),
                       "acumulada": freq.cumsum(), "relativa_acumulada": (freq / len(dados)).cumsum()})
display(tabela)

### Covariância e correlação de Pearson

**Quando usar:** quantificar associação linear entre duas variáveis quantitativas.

**Fórmula(s):**

$$s_{xy}=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{n-1},\qquad r=\frac{s_{xy}}{s_xs_y}$$

Correlação não implica causalidade; Pearson mede associação **linear**.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 3, 5, 4, 7], dtype=float)
print("Covariância amostral:", np.cov(x, y, ddof=1)[0, 1])
print("Correlação de Pearson:", stats.pearsonr(x, y).statistic)

---
# 2. Probabilidades

**Identificação:** eventos, contagens de casos, independência, condicionamento e atualização de probabilidades.

### Complemento, união e interseção

**Quando usar:** calcular a probabilidade de “não A” ou “A ou B”.

**Fórmula(s):**

$$P(A^c)=1-P(A)$$

$$P(A\cup B)=P(A)+P(B)-P(A\cap B)$$

Se A e B são mutuamente exclusivos, a interseção vale zero.

In [ ]:
p_a, p_b, p_intersecao = 0.4, 0.5, 0.2
print("P(não A):", 1 - p_a)
print("P(A ou B):", p_a + p_b - p_intersecao)

### Probabilidade condicional e independência

**Quando usar:** o enunciado contém “dado que”, “sabendo que” ou pergunta se eventos são independentes.

**Fórmula(s):**

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\quad (P(B)>0)$$

$$A\perp B\iff P(A\cap B)=P(A)P(B)$$

Não confundir **independentes** com **mutuamente exclusivos**.

In [ ]:
p_a, p_b, p_ab = 0.4, 0.5, 0.2
print("P(A|B):", p_ab / p_b)
print("São independentes?", np.isclose(p_ab, p_a * p_b))

### Probabilidade total e teorema de Bayes

**Quando usar:** há grupos/categorias mutuamente exclusivos e exaustivos; atualizar a probabilidade após uma evidência.

**Fórmula(s):**

$$P(B)=\sum_iP(B\mid A_i)P(A_i)$$

$$P(A_j\mid B)=\frac{P(B\mid A_j)P(A_j)}{\sum_iP(B\mid A_i)P(A_i)}$$

Os eventos Aᵢ devem formar uma partição e o denominador deve ser positivo.

In [ ]:
prioris = np.array([0.7, 0.3])
verossimilhancas = np.array([0.1, 0.8])  # P(B|A_i)
p_b = np.dot(prioris, verossimilhancas)
posteriores = prioris * verossimilhancas / p_b
print("P(B):", p_b)
print("P(A_i|B):", posteriores)

### Arranjos, permutações e combinações

**Quando usar:** contar resultados possíveis antes de calcular probabilidades.

**Fórmula(s):**

$$n!=n(n-1)\cdots1$$

$$A(n,k)=\frac{n!}{(n-k)!},\qquad \binom nk=\frac{n!}{k!(n-k)!}$$

**Arranjo:** ordem importa. **Combinação:** ordem não importa. **Permutação de n distintos:** n!.

In [ ]:
n, k = 8, 3
print("Permutações de n:", special.factorial(n, exact=True))
print("Arranjos sem repetição:", special.perm(n, k, exact=True))
print("Combinações sem repetição:", special.comb(n, k, exact=True))

---
# 3. Variáveis Aleatórias Discretas Unidimensionais

**Identificação:** valores isolados ou contagens; probabilidades pontuais e acumuladas.

**Atalho de interpretação (distribuições discretas):** `pmf(k)` = exatamente k; `cdf(k)` = até k (inclusive); `cdf(k-1)` = menos que k; `sf(k-1)` = pelo menos k; `sf(k)` = mais que k.

### Função de probabilidade e função acumulada

**Quando usar:** a variável assume valores discretos, com probabilidades associadas.

**Fórmula(s):**

$$p_X(x)=P(X=x),\qquad F_X(x)=P(X\leq x)=\sum_{t\leq x}p_X(t)$$

$$\sum_xp_X(x)=1$$

Verifique que todas as probabilidades são não negativas e somam 1.

In [ ]:
valores = np.array([0, 1, 2, 3])
probs = np.array([0.1, 0.2, 0.4, 0.3])
assert np.all(probs >= 0) and np.isclose(probs.sum(), 1)
print("P(X=2):", probs[valores == 2].sum())
print("P(X<=2):", probs[valores <= 2].sum())

### Esperança, variância e desvio-padrão discretos

**Quando usar:** a questão fornece valores possíveis e probabilidades, sem nomear uma distribuição padrão.

**Fórmula(s):**

$$E(X)=\sum_xxp_X(x),\qquad E(X^2)=\sum_xx^2p_X(x)$$

$$\operatorname{Var}(X)=E(X^2)-[E(X)]^2,\qquad \sigma_X=\sqrt{\operatorname{Var}(X)}$$

Probabilidades podem ser decimais (`0.25`) ou frações Python (`1/4`).

In [ ]:
valores = np.array([0, 1, 2, 3], dtype=float)
probs = np.array([1/10, 2/10, 4/10, 3/10], dtype=float)
assert np.isclose(probs.sum(), 1)
esperanca = np.dot(valores, probs)
variancia = np.dot(valores**2, probs) - esperanca**2
print("Esperança:", esperanca)
print("Variância:", variancia)
print("Desvio-padrão:", np.sqrt(max(variancia, 0)))

### Transformações lineares

**Quando usar:** calcular esperança e variância de Y=aX+b.

**Fórmula(s):**

$$E(aX+b)=aE(X)+b$$

$$\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X)$$

A constante b desloca a média, mas não altera a variância.

In [ ]:
media_x, var_x = 4, 9
a, b = 2, 5
print("E(Y):", a * media_x + b)
print("Var(Y):", a**2 * var_x)

### Bernoulli

**Quando usar:** uma única tentativa, com sucesso (1) ou fracasso (0).

**Fórmula(s):**

$$P(X=x)=p^x(1-p)^{1-x},\quad x\in\{0,1\}$$

$$E(X)=p,\qquad \operatorname{Var}(X)=p(1-p)$$

Parâmetro: probabilidade p de sucesso.

In [ ]:
probabilidade = 0.2

p = probabilidade
d = stats.bernoulli(p)
print("P(X=1):", d.pmf(1), "E(X):", d.mean(), "Var(X):", d.var())

### Binomial

**Quando usar:** n tentativas independentes, duas possibilidades em cada uma e p constante.

**Fórmula(s):**

$$P(X=k)=\binom nkp^k(1-p)^{n-k}$$

$$E(X)=np,\qquad \operatorname{Var}(X)=np(1-p)$$

**Sem reposição** em população finita geralmente não é binomial; veja hipergeométrica.

In [ ]:
instancias = 5
probabilidade = 10/20
instancias_sucesso = 2

n, p, k = instancias, probabilidade, instancias_sucesso
d = stats.binom(n=n, p=p)
print("Exatamente k:", d.pmf(k))
print("No máximo k:", d.cdf(k))
print("Menos que k:", d.cdf(k-1))
print("Pelo menos k:", d.sf(k-1))
print("Mais que k:", d.sf(k))
print("Média:", d.mean())
print("Variância:", d.var())

### Poisson

**Quando usar:** contagem de ocorrências em intervalo, sob modelo de taxa média constante e ocorrências independentes.

**Fórmula(s):**

$$P(X=k)=e^{-\lambda}\frac{\lambda^k}{k!}$$

$$E(X)=\operatorname{Var}(X)=\lambda$$

Se a taxa é por unidade e o intervalo tem duração t, use λ = taxa × t.

In [ ]:
taxa_media = 3
periodo = 6
k = 8
lam = taxa_media * periodo
d = stats.poisson(mu=lam)
print(f"P(X={k}): {d.pmf(k)} (chance de exatamente {k})")
print(f"P(X>={k}): {d.sf(k-1)} (chance de no mínimo {k})")
print(f"P(X<={k}): {d.cdf(k)} (chance de  no máximo {k})")
print("Esperança:", d.mean())
print("Variância: ", d.var())
print("Na Distribuição de Poisson, tanto a variância quanto a esperança são iguais a lambda!")

### Geométrica

**Quando usar:** contar o número de tentativas até o primeiro sucesso, com ensaios independentes e p constante.

**Fórmula(s):**

$$P(X=k)=(1-p)^{k-1}p,\quad k=1,2,\ldots$$

$$E(X)=\frac1p,\qquad \operatorname{Var}(X)=\frac{1-p}{p^2}$$

**Convenção:** SciPy conta tentativas a partir de 1. Alguns livros contam fracassos a partir de 0.

In [ ]:
p = 0.8
k = 3
d = stats.geom(p)
print(f"Primeiro sucesso na tentativa {k}: {d.pmf(k)}")
print(f"Até {k} tentativas: {d.cdf(k)}")
print(f"Média: {d.mean()}\nVariância: {d.var()}")

print("Sucesso exatamente em k para vários k:")

for k in range(1, k+1):
    print(f"k={k}: {d.pmf(k)}")

### Hipergeométrica

**Quando usar:** amostragem **sem reposição** de população finita, com sucessos e fracassos.

**Fórmula(s):**

$$P(X=k)=\frac{\binom Kk\binom{N-K}{n-k}}{\binom Nn}$$

$$E(X)=n\frac KN$$

N = população; K = sucessos na população; n = tamanho da amostra. SciPy usa `M=N, n=K, N=n_amostra`.

In [ ]:
populacao = 5
sucessos_pop = 3
tamanho_amostra = 30
k = 8
d = stats.hypergeom(M=populacao, n=sucessos_pop, N=tamanho_amostra)
print(f"""P(X={k}): {d.pmf(k)}
P(X<={k}): {d.cdf(k)} (probabilidade acumulada)
Média: {d.mean()}""")

---
# 4. Variáveis Aleatórias Contínuas Unidimensionais

**Identificação:** densidades, integrais, probabilidades de intervalos, percentis e modelos contínuos. Para variável contínua, P(X = a) = 0.

### Densidade, acumulada e intervalos

**Quando usar:** uma função densidade f(x) é fornecida e pede-se probabilidade em intervalo.

**Fórmula(s):**

$$f(x)\geq0,\qquad \int_{-\infty}^{\infty}f(x)\,dx=1$$

$$F(x)=\int_{-\infty}^xf(t)\,dt,\qquad P(a\leq X\leq b)=F(b)-F(a)$$

Limites inclusivos/exclusivos não alteram probabilidades em variáveis contínuas.

In [ ]:
# Exemplo: f(x)=2x em [0,1], zero fora
def f(x):
    return 2*x if 0 <= x <= 1 else 0.0

a, b = 0.2, 0.8
prob, erro_numerico = integrate.quad(f, a, b)
print("P(a<=X<=b):", prob)
print("Integral total:", integrate.quad(f, 0, 1)[0])

### Esperança e variância contínuas

**Quando usar:** a densidade é conhecida e não há distribuição padrão diretamente aplicável.

**Fórmula(s):**

$$E(X)=\int_{-\infty}^{\infty}xf(x)\,dx$$

$$\operatorname{Var}(X)=\int_{-\infty}^{\infty}x^2f(x)\,dx-[E(X)]^2$$

Integração numérica retorna aproximações; confirme o suporte e se f integra 1.

In [ ]:
def f(x):
    return 2*x if 0 <= x <= 1 else 0.0
media = integrate.quad(lambda x: x*f(x), 0, 1)[0]
segundo_momento = integrate.quad(lambda x: x*x*f(x), 0, 1)[0]
print("Esperança:", media)
print("Variância:", segundo_momento - media**2)

### Uniforme contínua

**Quando usar:** todos os pontos de um intervalo [a,b] têm a mesma densidade.

**Fórmula(s):**

$$f(x)=\frac1{b-a},\quad a\leq x\leq b$$

$$E(X)=\frac{a+b}{2},\qquad \operatorname{Var}(X)=\frac{(b-a)^2}{12}$$

SciPy: `loc=a`, `scale=b-a` (não é o extremo superior).

In [ ]:
a, b = 2, 8
d = stats.uniform(loc=a, scale=b-a)
print("P(3<=X<=5):", d.cdf(5) - d.cdf(3))
print("Média, variância:", d.mean(), d.var())

### Normal e padronização Z

**Quando usar:** valores contínuos aproximadamente normais; cálculo de caudas e percentis.

**Fórmula(s):**

$$X\sim N(\mu,\sigma^2),\qquad Z=\frac{X-\mu}{\sigma}\sim N(0,1)$$

$$f(x)=\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

SciPy: `scale` recebe **desvio-padrão σ**, não variância σ². `ppf(q)` é o quantil com probabilidade acumulada q.

In [ ]:
mu, sigma, x = 100, 15, 120
d = stats.norm(loc=mu, scale=sigma)
print("P(X<=x):", d.cdf(x))
print("P(X>x):", d.sf(x))
print("P(90<=X<=120):", d.cdf(120) - d.cdf(90))
print("Percentil 95:", d.ppf(0.95))
print("Z de x:", (x-mu)/sigma)

### Exponencial

**Quando usar:** tempo de espera entre eventos de processo de Poisson com taxa λ.

**Fórmula(s):**

$$f(x)=\lambda e^{-\lambda x},\quad x\geq0$$

$$F(x)=1-e^{-\lambda x},\quad E(X)=\frac1\lambda,\quad \operatorname{Var}(X)=\frac1{\lambda^2}$$

SciPy: `scale=1/lambda` (não passar λ diretamente como scale).

In [ ]:
taxa, t = 0.5, 3
D = stats.expon(scale=1/taxa)
print("P(X<=t):", D.cdf(t))
print("P(X>t):", D.sf(t))
print("Média, variância:", D.mean(), D.var())

### Quantis e probabilidades inversas

**Quando usar:** a questão fornece uma probabilidade e pede o valor de corte.

**Fórmula(s):**

$$F(x_q)=q\qquad\Longrightarrow\qquad x_q=F^{-1}(q)$$

`ppf(q)` é o inverso da acumulada; `isf(q)` encontra o corte cuja cauda superior vale q.

In [ ]:
d = stats.norm(loc=50, scale=10)
print("Valor abaixo do qual estão 90%:", d.ppf(0.90))
print("Valor com 5% acima:", d.isf(0.05))

---
# 5. Testes de Hipótese

**Roteiro obrigatório:** (1) escrever H₀ e H₁; (2) identificar parâmetro e pressupostos; (3) escolher teste e cauda; (4) calcular estatística e valor-p; (5) comparar com α; (6) concluir **no contexto**. **Não rejeitar H₀** não significa demonstrar que H₀ é verdadeira.

**Regra:** rejeitar H₀ quando `valor-p <= alfa`. Use teste unilateral somente quando a hipótese alternativa for direcional **antes** de olhar os dados.

### Teste Z para média (σ populacional conhecido)

**Quando usar:** testar média populacional com σ conhecido e população normal ou aproximação normal justificada.

**Fórmula(s):**

$$H_0:\mu=\mu_0,\qquad Z=\frac{\bar X-\mu_0}{\sigma/\sqrt n}$$

Escolha `alternativa` = `"two-sided"`, `"greater"` ou `"less"`.

In [ ]:
media_amostral, mu0, sigma_pop, n = 53, 50, 10, 40
alternativa, alfa = "two-sided", 0.05
z = (media_amostral - mu0) / (sigma_pop / np.sqrt(n))
p = (2*stats.norm.sf(abs(z)) if alternativa == "two-sided" else
     stats.norm.sf(z) if alternativa == "greater" else stats.norm.cdf(z))
print("Z:", z, "valor-p:", p, "Rejeitar H0?", p <= alfa)

### Teste t para uma média (σ desconhecido)

**Quando usar:** testar média com desvio-padrão populacional desconhecido; independência e normalidade/aproximação adequadas.

**Fórmula(s):**

$$T=\frac{\bar X-\mu_0}{s/\sqrt n},\qquad gl=n-1$$

`ttest_1samp` aceita observações brutas e calcula estatística, valor-p e graus de liberdade.

In [ ]:
dados = np.array([49, 52, 54, 51, 55, 53, 50, 56], dtype=float)
mu0, alternativa, alfa = 50, "greater", 0.05
res = stats.ttest_1samp(dados, popmean=mu0, alternative=alternativa)
print("t:", res.statistic, "valor-p:", res.pvalue)
print("Rejeitar H0?", res.pvalue <= alfa)

### Intervalo de confiança para média

**Quando usar:** estimar média com incerteza (não é teste por si só).

**Fórmula(s):**

$$IC_{1-\alpha}(\mu)=\bar x\pm t_{1-\alpha/2,\,n-1}\frac{s}{\sqrt n}$$

Forma t para σ desconhecido; pressupostos análogos ao teste t de uma amostra.

In [ ]:
dados = np.array([49, 52, 54, 51, 55, 53, 50, 56], dtype=float)
confianca = 0.95
n = len(dados)
media, se = np.mean(dados), stats.sem(dados)
intervalo = stats.t.interval(confianca, df=n-1, loc=media, scale=se)
print("IC para média:", intervalo)

### Teste t para duas amostras independentes

**Quando usar:** comparar médias de dois grupos independentes.

**Fórmula(s):**

$$H_0:\mu_1=\mu_2$$

$$t_{Welch}=\frac{\bar x_1-\bar x_2}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$$

**Welch** (`equal_var=False`) não exige variâncias populacionais iguais; exige independência e condições apropriadas.

In [ ]:
grupo1 = np.array([10, 11, 12, 9, 13], dtype=float)
grupo2 = np.array([8, 9, 10, 11, 9], dtype=float)
res = stats.ttest_ind(grupo1, grupo2, equal_var=False, alternative="two-sided")
print("t de Welch:", res.statistic, "valor-p:", res.pvalue)

### Teste t pareado

**Quando usar:** mesmos indivíduos antes/depois ou observações em pares.

**Fórmula(s):**

$$d_i=x_i-y_i,\qquad T=\frac{\bar d}{s_d/\sqrt n}$$

As diferenças entre pares devem satisfazer as condições do teste t de uma amostra.

In [ ]:
antes = np.array([80, 75, 90, 85, 70], dtype=float)
depois = np.array([78, 73, 87, 82, 69], dtype=float)
res = stats.ttest_rel(antes, depois, alternative="two-sided")
print("t pareado:", res.statistic, "valor-p:", res.pvalue)

### Teste para uma proporção (aproximação Z)

**Quando usar:** testar proporção populacional em ensaios Bernoulli independentes com aproximação normal válida.

**Fórmula(s):**

$$H_0:p=p_0,\qquad Z=\frac{\hat p-p_0}{\sqrt{p_0(1-p_0)/n}}$$

Para amostras pequenas, prefira o **teste binomial exato**; abaixo ambos são mostrados.

In [ ]:
sucessos, n, p0 = 56, 100, 0.5
p_hat = sucessos / n
z = (p_hat-p0) / np.sqrt(p0*(1-p0)/n)
p_z = 2*stats.norm.sf(abs(z))
p_exato = stats.binomtest(sucessos, n, p0, alternative="two-sided").pvalue
print("Proporção observada:", p_hat, "Z:", z)
print("valor-p aproximado:", p_z, "valor-p exato:", p_exato)

### Qui-quadrado de aderência

**Quando usar:** comparar frequências observadas com frequências esperadas por um modelo categórico.

**Fórmula(s):**

$$\chi^2=\sum_i\frac{(O_i-E_i)^2}{E_i}$$

Frequências esperadas não devem ser muito pequenas; se parâmetros forem estimados dos dados, os graus de liberdade podem mudar.

In [ ]:
observadas = np.array([18, 22, 30, 30])
esperadas = np.array([25, 25, 25, 25])
res = stats.chisquare(f_obs=observadas, f_exp=esperadas)
print("Qui-quadrado:", res.statistic, "valor-p:", res.pvalue)

### Qui-quadrado de independência

**Quando usar:** verificar associação entre duas variáveis categóricas em tabela de contingência.

**Fórmula(s):**

$$E_{ij}=\frac{(\text{total da linha i})(\text{total da coluna j})}{n}$$

$$\chi^2=\sum_{i,j}\frac{(O_{ij}-E_{ij})^2}{E_{ij}}$$

Exige observações independentes; para tabela 2×2 com contagens pequenas, considere Fisher exato.

In [ ]:
tabela = np.array([[20, 10], [12, 18]])
qui2, p, gl, esperadas = stats.chi2_contingency(tabela, correction=False)
print("Qui-quadrado:", qui2, "gl:", gl, "valor-p:", p)
print("Frequências esperadas:\n", esperadas)
print("Fisher exato (2x2):", stats.fisher_exact(tabela).pvalue)

### Erros tipo I, tipo II e poder

**Quando usar:** interpretar decisões e riscos de testes.

**Fórmula(s):**

$$\alpha=P(\text{rejeitar }H_0\mid H_0\text{ verdadeira})$$

$$\beta=P(\text{não rejeitar }H_0\mid H_1\text{ verdadeira}),\qquad \text{poder}=1-\beta$$

Poder depende de tamanho amostral, efeito real, variabilidade e regra do teste. Não há função universal sem especificar esses elementos.

In [ ]:
alfa = 0.05
beta_exemplo = 0.20
print("Nível de significância:", alfa)
print("Poder ilustrativo:", 1-beta_exemplo)

---
# 6. Modelo Básico de Regressão Linear

**Identificação:** relação entre variável explicativa x e resposta y; ajuste por mínimos quadrados; interpretação de coeficientes, resíduos, R² e inferência.

**Pressupostos usuais para inferência OLS:** linearidade na média condicional, exogeneidade, observações adequadamente independentes, variância constante para erros-padrão clássicos e normalidade dos erros para testes exatos em pequenas amostras. Verifique os pressupostos exigidos no curso.

### Modelo e coeficientes de mínimos quadrados

**Quando usar:** ajustar reta a pares (x,y).

**Fórmula(s):**

$$Y_i=\beta_0+\beta_1x_i+\varepsilon_i$$

$$\hat\beta_1=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2},\qquad\hat\beta_0=\bar y-\hat\beta_1\bar x$$

Inclua constante explicitamente com `sm.add_constant`. Inclinação: variação prevista de y por unidade de x.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
X = sm.add_constant(x)
modelo = sm.OLS(y, X).fit()
print("Intercepto, inclinação:", modelo.params)
print("R²:", modelo.rsquared)

### Predição, resíduos e soma dos quadrados

**Quando usar:** prever valores e analisar erros do ajuste.

**Fórmula(s):**

$$\hat y_i=\hat\beta_0+\hat\beta_1x_i,\qquad e_i=y_i-\hat y_i$$

$$SQE=\sum_i e_i^2$$

Predições fora da faixa observada são extrapolações e merecem cautela.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
modelo = sm.OLS(y, sm.add_constant(x)).fit()
print("Valores ajustados:", modelo.fittedvalues)
print("Resíduos:", modelo.resid)
print("SQE:", np.sum(modelo.resid**2))
x_novo = 7
print("Previsão para x=7:", modelo.predict([1, x_novo])[0])

### Coeficiente de determinação R²

**Quando usar:** medir a proporção da variação amostral de y explicada pela reta com intercepto.

**Fórmula(s):**

$$R^2=1-\frac{SQE}{SQT},\qquad SQT=\sum_i(y_i-\bar y)^2$$

Para regressão simples com intercepto, R² = r². Isso não prova causalidade.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
modelo = sm.OLS(y, sm.add_constant(x)).fit()
print("R²:", modelo.rsquared)
print("R² ajustado:", modelo.rsquared_adj)
print("r² de Pearson:", stats.pearsonr(x, y).statistic**2)

### Erro-padrão residual e teste t dos coeficientes

**Quando usar:** quantificar dispersão residual e testar H₀: β₁ = 0.

**Fórmula(s):**

$$s_e=\sqrt{\frac{SQE}{n-2}}$$

$$t=\frac{\hat\beta_1-0}{EP(\hat\beta_1)}$$

A fórmula n−2 vale para regressão simples **com intercepto**. Valores-p padrão dependem das hipóteses inferenciais.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
modelo = sm.OLS(y, sm.add_constant(x)).fit()
print("Erro-padrão residual:", np.sqrt(modelo.mse_resid))
print("t dos coeficientes:", modelo.tvalues)
print("Valores-p dos coeficientes:", modelo.pvalues)
print("IC 95% dos coeficientes:\n", modelo.conf_int(alpha=0.05))

### Intervalos de confiança e de previsão

**Quando usar:** estimar a resposta média em x₀ ou prever uma nova observação em x₀.

**Fórmula(s):**

$$IC\text{ da média: incerteza de }E(Y\mid X=x_0)$$

$$IP\text{ individual: inclui variabilidade de uma nova observação}$$

Intervalo de previsão individual é geralmente mais largo que intervalo de confiança da média.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
modelo = sm.OLS(y, sm.add_constant(x)).fit()
x0 = 4.5
predicao = modelo.get_prediction(exog=np.array([[1.0, x0]]))
display(predicao.summary_frame(alpha=0.05))

### Gráfico de dispersão, reta e resíduos

**Quando usar:** inspecionar visualmente relação linear, ajuste e possíveis padrões nos resíduos.

**Fórmula(s):**

$$e_i=y_i-\hat y_i$$

Procure curvatura, heterocedasticidade, observações atípicas e extrapolação.

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 3, 5, 4, 6, 8], dtype=float)
modelo = sm.OLS(y, sm.add_constant(x)).fit()
plt.scatter(x, y, label="Dados")
plt.plot(x, modelo.fittedvalues, label="Reta ajustada")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()
plt.scatter(modelo.fittedvalues, modelo.resid)
plt.axhline(0, linestyle="--")
plt.xlabel("Valores ajustados"); plt.ylabel("Resíduos"); plt.show()

---
# 7. Dicionário de notações

> **Importante:** uma mesma letra pode representar coisas diferentes conforme o assunto. A interpretação deve considerar a fórmula em que aparece.

## Letras gregas

| Símbolo | Nome | Significado frequente |
|---|---|---|
| $\alpha$ | alfa | nível de significância; também parâmetro em alguns modelos |
| $\beta$ | beta | probabilidade de erro tipo II; coeficiente de regressão (depende do contexto) |
| $\beta_0,\beta_1$ | beta zero, beta um | intercepto e inclinação populacionais |
| $\varepsilon$ | épsilon | termo de erro na regressão |
| $\lambda$ | lambda | taxa ou média da distribuição de Poisson; taxa da exponencial |
| $\mu$ | mi | média populacional / esperança |
| $\pi$ | pi | constante matemática ≈ 3,14159; por vezes proporção populacional |
| $\rho$ | rô | correlação populacional |
| $\sigma$ | sigma | desvio-padrão populacional |
| $\sigma^2$ | sigma ao quadrado | variância populacional |
| $\Sigma$ | sigma maiúsculo | somatório |
| $\chi^2$ | qui-quadrado | estatística ou distribuição qui-quadrado |

## Letras latinas e símbolos estatísticos

| Símbolo | Significado frequente |
|---|---|
| $X,Y$ | variáveis aleatórias |
| $x_i,y_i$ | valores observados |
| $n$ | tamanho da amostra ou número de tentativas |
| $N$ | tamanho da população (ou parâmetro de modelo específico) |
| $k$ | contagem de sucessos ou índice |
| $p$ | probabilidade de sucesso ou proporção populacional |
| $\hat p$ | proporção amostral |
| $\bar x$ | média amostral |
| $s,s^2$ | desvio-padrão e variância amostrais |
| $E(X)$ | esperança / valor esperado |
| $\operatorname{Var}(X)$ | variância de X |
| $f(x)$ | densidade (contínua) ou função definida no contexto |
| $p_X(x)$ | função de probabilidade (discreta) |
| $F(x)$ | função de distribuição acumulada |
| $P(A)$ | probabilidade do evento A |
| $P(A\mid B)$ | probabilidade de A dado B |
| $H_0,H_1$ | hipóteses nula e alternativa |
| $EP$ | erro-padrão |
| $gl$ | graus de liberdade |
| $r$ | correlação amostral |
| $R^2$ | coeficiente de determinação |
| $\hat y$ | valor previsto/ajustado de y |
| $e_i$ | resíduo amostral |
| $\hat\theta$ | estimativa do parâmetro $\theta$ |

## Operadores e relações

| Símbolo | Significado |
|---|---|
| $\sum$ | somatório |
| $\prod$ | produtório |
| $\int$ | integral |
| $\sqrt{a}$ | raiz quadrada |
| $n!$ | fatorial |
| $\binom nk$ | combinações de n elementos tomados k a k |
| $\cap$ | interseção: A **e** B |
| $\cup$ | união: A **ou** B (inclusive) |
| $A^c$ | complemento de A |
| $\mid$ | “dado que”, em probabilidade condicional |
| $\perp$ | independência (quando usado entre eventos/variáveis) |
| $\in$ | pertence |
| $\sim$ | segue a distribuição indicada |
| $\hat{\ }$ | acento chapéu: estimativa ou valor ajustado |
| $\le,\ge$ | menor/maior ou igual |
| $\approx$ | aproximadamente igual |
| $\infty$ | infinito |

## Atalhos Python (SciPy)

| Função | Interpretação |
|---|---|
| `d.pmf(k)` | P(X = k), para distribuições discretas |
| `d.pdf(x)` | densidade em x, **não** probabilidade pontual |
| `d.cdf(x)` | P(X ≤ x) |
| `d.sf(x)` | P(X > x); costuma ser mais preciso que `1 - cdf(x)` |
| `d.ppf(q)` | quantil com acumulada q |
| `d.isf(q)` | quantil da cauda superior q |
| `d.mean()` | esperança |
| `d.var()` | variância |
| `d.std()` | desvio-padrão |
| `np.mean(dados)` | média aritmética dos dados (não é esperança ponderada arbitrária) |
| `np.dot(valores, probabilidades)` | esperança discreta personalizada |

## Checklist de prova

1. **Identifique:** dados brutos, eventos, distribuição, teste ou regressão?
2. **Confira as hipóteses:** independência, reposição, normalidade, amostragem, variância conhecida?
3. **Defina o que pede:** exatamente, até, pelo menos, intervalo, quantil, média, valor-p?
4. **Ajuste os parâmetros** na célula do tópico correspondente.
5. **Interprete** o número obtido (probabilidade, percentual, unidades, decisão estatística).

**Nota:** códigos exibem aproximações numéricas; quando a prova exigir desenvolvimento algébrico, mostre também a fórmula e a substituição dos valores.